In [0]:
%sql
show databases

In [0]:
%sql
use pnp_gold

#### create tables

In [0]:
%sql
CREATE TABLE IF NOT EXISTS pnp_gold.prices
USING DELTA
LOCATION 'abfss://gold@pnpbyedge.dfs.core.windows.net/prices/';

In [0]:
%sql
CREATE TABLE IF NOT EXISTS pnp_gold.prices_current
USING DELTA
AS
SELECT * FROM pnp_gold.prices WHERE 1=0;

In [0]:
%sql
CREATE TABLE IF NOT EXISTS pnp_gold.watermark
USING DELTA
LOCATION 'abfss://gold@pnpbyedge.dfs.core.windows.net/Watermark/';

-- This now points at the SAME folder your
-- notebook already writes watermark data to!

In [0]:
%sql
SHOW TABLES IN asc_pnp.pnp_gold;

In [0]:
%sql
DESCRIBE TABLE pnp_gold.prices;
DESCRIBE TABLE pnp_gold.prices_current;
DESCRIBE TABLE pnp_gold.watermark;

## merge table price into current price to know live price 

In [0]:
%sql
merge into pnp_gold.prices_current AS target

using (

    select * from pnp_gold.prices
    where date = current_date()
) as source

on target.sku = source.sku 
and target.retailer_name = source.retailer_name 

when matched and (
    target.effective_price != source.effective_price    
    or 
    target.is_promo != source.is_promo
    or 
    target.promo_price IS DISTINCT FROM source.promo_price

)
then update set *
when not matched  then insert *;

In [0]:
%sql
SELECT (SELECT COUNT(*) FROM pnp_gold.prices) AS total_rows_price, (SELECT COUNT(*) FROM pnp_gold.prices_current) AS total_rows_prices_current, (SELECT COUNT(*) FROM pnp_gold.watermark) AS total_rows_watermark

In [0]:
%sql
-- See data split by date
SELECT date, COUNT(*) AS rows
FROM pnp_gold.prices
GROUP BY date
ORDER BY date;

In [0]:
%sql
-- Price history for one product
SELECT date, retailer_name, effective_price
FROM pnp_gold.prices
WHERE sku = 'SKULDL107842'
ORDER BY date;

In [0]:
%sql

select retailer_name, sku, product_name, effective_price, date 
from pnp_gold.prices_current
limit 20;

In [0]:
%sql
select * from pnp_gold.watermark

In [0]:
%sql
DESCRIBE CATALOG asc_pnp;
DESCRIBE SCHEMA asc_pnp.pnp_gold;

In [0]:
%sql
SHOW GRANTS ON CATALOG asc_pnp;
SHOW GRANTS ON SCHEMA asc_pnp.pnp_gold;
SHOW GRANTS ON TABLE asc_pnp.pnp_gold.prices;

In [0]:
%sql
